# Diabetic Retinopathy — Training Control Panel

- Dataset: `data/btgraham-300/`; original official split membership is preserved.
- Current fast strategy: `undersampled`. Training keeps at most **2,500 examples per class**; validation and test are also reduced to approximate a **70:10:20** train/validation/test distribution.
- Run `prepare_undersampled_split.ipynb` once before training. It writes one manifest used by EfficientNet, weighted EfficientNet, DenseNet, fusion, and the evaluator.
- Expected sizes from the project's known counts: **9,024 train / 1,289 validation / 2,578 test**. Trust the actual counts printed by the preparation notebook if they differ.
- Undersampled runs use separate names such as `models/efficientnet_undersampled/`; older full-data runs/checkpoints are not overwritten.
- Metrics represent the fixed sampled validation/test subsets, not the complete official validation/test splits. Start training only by manually executing the corresponding training cell.


## 1. Environment and device check

In [1]:
from configs import tensorflow_setup as _tensorflow_setup

import sys
import numpy as np
import tensorflow as tf

_tensorflow_setup.configure_gpu_memory_growth(tf)

import tensorflow_datasets as tfds

print("Python:", sys.version.split()[0])
print("CUDA DLL directory:", _tensorflow_setup.CUDA_DLL_DIRECTORY)
print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "not exposed")
print("NumPy:", np.__version__)
print("TFDS:", tfds.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("EfficientNetB0 API:", hasattr(tf.keras.applications, "EfficientNetB0"))
print("Seed utility API:", hasattr(tf.keras.utils, "set_random_seed"))
from configs.config import TRAINING_SAMPLING_MODE, UNDERSAMPLE_MAX_PER_CLASS, SEED, training_model_name
print("Training sampling mode:", TRAINING_SAMPLING_MODE)
if TRAINING_SAMPLING_MODE == "undersampled":
    print("Maximum training examples per class:", UNDERSAMPLE_MAX_PER_CLASS)


Python: 3.10.20
CUDA DLL directory: C:\Users\kawal\miniconda3\envs\dr_tf\Library\bin
TensorFlow: 2.10.1
Keras: 2.10.0
NumPy: 1.23.5
TFDS: 4.8.3
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
EfficientNetB0 API: True
Seed utility API: True
Training sampling mode: undersampled
Maximum training examples per class: 2500


## 2. Data-loader smoke test

In [2]:
from scripts.data_loader import inspect_dataset, make_datasets, get_training_sampling_summary

inspect_dataset()
sampling_summary = get_training_sampling_summary(
    seed=SEED,
    training_sampling_mode=TRAINING_SAMPLING_MODE,
)
print("Original training class counts:", sampling_summary["original_training_class_counts"])
print("Retained training class counts:", sampling_summary["training_class_counts"])
print("Training examples retained:", sampling_summary["retained_training_examples"])
print("Validation examples retained:", sampling_summary.get("validation_examples_after_sampling"))
print("Test examples retained:", sampling_summary.get("test_examples_after_sampling"))
print("Training subset SHA-256:", sampling_summary.get("training_subset_sha256"))
print("Whole split-manifest SHA-256:", sampling_summary.get("split_manifest_sha256"))
if TRAINING_SAMPLING_MODE == "undersampled" and not sampling_summary.get("split_manifest_sha256"):
    raise RuntimeError("No fixed split manifest found. Run prepare_undersampled_split.ipynb first.")
train_ds, validation_ds, test_ds = make_datasets(
    batch_size=4,
    seed=SEED,
    training_sampling_mode=TRAINING_SAMPLING_MODE,
)
images, labels = next(iter(train_ds))

print("Image batch shape:", images.shape)
print("Image dtype:", images.dtype)
print("Pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))
print("Labels in first batch:", labels.numpy())

Dataset directory: D:\M L\Diabetic_Retinopathy_Detection_mini_project\data\250K
Dataset name:      diabetic_retinopathy_detection
Dataset version:   3.0.0
  sample: 10 examples
  train: 35,126 examples
  validation: 10,906 examples
  test: 42,670 examples
Original training class counts: [25810, 2443, 5292, 873, 708]
Retained training class counts: [2500, 2443, 2500, 873, 708]
Training examples retained: 9024
Validation examples retained: 1289
Test examples retained: 2578
Training subset SHA-256: 0e2d8969d3ea34a2466bf438d91634a95d8828d65d1e9d652c407e6b865b8dcb
Whole split-manifest SHA-256: d8a125deedf71c91bc844c3cbe64e0d2a4ee144dcf604093e899a56401d7d471
Using fixed 70:10:20 subset manifest: train=9,024, validation=1,289, test=2,578; SHA-256=d8a125deedf7…
Train     : 9,024 selected / 35,126 official; class counts=[2500, 2443, 2500, 873, 708]
Validation: 1,289 selected / 10,906 official; class counts=[961, 85, 187, 28, 28]
Test      : 2,578 selected / 42,670 official; class counts=[1897, 

## 3. EfficientNet — model initialization preview

In [3]:
from scripts.train_efficientnet import train_efficientnet

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False

if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    print("1. Stop without training")
    print("2. Allow potentially slow CPU training")
    cpu_choice = int(input("Select CPU action [1-2]: ").strip())
    if cpu_choice == 2:
        allow_cpu = True
    elif cpu_choice != 1:
        raise ValueError("Choose 1 to stop or 2 to allow CPU training.")

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, then rerun this cell.")
else:
    print("1. Start a new scratch run")
    print("2. Resume the latest checkpointed run")
    run_choice = int(input("Select run mode [1-2]: ").strip())
    if run_choice not in (1, 2):
        raise ValueError("Choose 1 for scratch or 2 for resume.")
    run_mode = ("scratch", "resume")[run_choice - 1]

    batch_choices = (4, 8, 16, 32)
    for index, value in enumerate(batch_choices, start=1):
        print(f"{index}. Batch size {value}")
    batch_choice = int(input("Select batch size [1-4]: ").strip())
    if not 1 <= batch_choice <= len(batch_choices):
        raise ValueError("Choose one of the displayed batch-size options.")
    batch_size = batch_choices[batch_choice - 1]

    epochs_text = input("Enter total target epochs as a whole number [10]: ").strip()
    target_epochs = int(epochs_text or 10)
    if target_epochs < 1:
        raise ValueError("Target epochs must be at least 1.")

    learning_rate_text = input("Enter learning rate [0.001]: ").strip()
    learning_rate = float(learning_rate_text or 0.001)
    if learning_rate <= 0:
        raise ValueError("Learning rate must be greater than 0.")

    efficientnet_result = train_efficientnet(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate,
        training_sampling_mode=TRAINING_SAMPLING_MODE
    )


Model: "efficientnet_b0_dr"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 retinal_image (InputLayer)  [(None, 512, 512, 3)]     0         
                                                                 
 efficientnetb0 (Functional)  (None, 16, 16, 1280)     4049571   
                                                                 
 global_average_pooling (Glo  (None, 1280)             0         
 balAveragePooling2D)                                            
                                                                 
 head_dropout (Dropout)      (None, 1280)              0         
                                                                 
 dr_severity (Dense)         (None, 5)                 6405      
                                                                 
Total params: 4,055,976
Trainable params: 6,405
Non-trainable params: 4,049,571
__________________________________

## 4. EfficientNet — manually controlled training
Choose `scratch` for a new isolated run or `resume` to continue the newest valid checkpoint. The epoch input is the **total target epoch**, not the number of additional epochs.

In [4]:
from scripts.train_efficientnet import train_efficientnet

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False

if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    print("1. Stop without training")
    print("2. Allow potentially slow CPU training")
    cpu_choice = int(input("Select CPU action [1-2]: ").strip())
    if cpu_choice == 2:
        allow_cpu = True
    elif cpu_choice != 1:
        raise ValueError("Choose 1 to stop or 2 to allow CPU training.")

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, then rerun this cell.")
else:
    print("1. Start a new scratch run")
    print("2. Resume the latest checkpointed run")
    run_choice = int(input("Select run mode [1-2]: ").strip())
    if run_choice not in (1, 2):
        raise ValueError("Choose 1 for scratch or 2 for resume.")
    run_mode = ("scratch", "resume")[run_choice - 1]

    batch_choices = (4, 8, 16, 32)
    for index, value in enumerate(batch_choices, start=1):
        print(f"{index}. Batch size {value}")
    batch_choice = int(input("Select batch size [1-4]: ").strip())
    if not 1 <= batch_choice <= len(batch_choices):
        raise ValueError("Choose one of the displayed batch-size options.")
    batch_size = batch_choices[batch_choice - 1]

    epochs_text = input("Enter total target epochs as a whole number [10]: ").strip()
    target_epochs = int(epochs_text or 10)
    if target_epochs < 1:
        raise ValueError("Target epochs must be at least 1.")

    learning_rate_text = input("Enter learning rate [0.001]: ").strip()
    learning_rate = float(learning_rate_text or 0.001)
    if learning_rate <= 0:
        raise ValueError("Learning rate must be greater than 0.")

    efficientnet_result = train_efficientnet(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate,
        training_sampling_mode=TRAINING_SAMPLING_MODE
    )


Training device(s): /physical_device:GPU:0
Starting a new, separate run: efficientnet_undersampled/run_001
Using fixed 70:10:20 subset manifest: train=9,024, validation=1,289, test=2,578; SHA-256=d8a125deedf7…
Train     : 9,024 selected / 35,126 official; class counts=[2500, 2443, 2500, 873, 708]
Validation: 1,289 selected / 10,906 official; class counts=[961, 85, 187, 28, 28]
Test      : 2,578 selected / 42,670 official; class counts=[1897, 184, 380, 59, 58]
Training sampling mode: undersampled; cap: 2500
Input size: 300x300; classes: 5
Starting at epoch 0; target total epochs: 10
Epoch 1/10
282/282 [==============================] - 60s 159ms/step - loss: 1.3610 - accuracy: 0.3686 - val_loss: 1.1904 - val_accuracy: 0.3794
Epoch 2/10
282/282 [==============================] - 45s 156ms/step - loss: 1.2499 - accuracy: 0.4227 - val_loss: 1.2665 - val_accuracy: 0.3351
Epoch 3/10
282/282 [==============================] - 42s 147ms/step - loss: 1.2247 - accuracy: 0.4354 - val_loss: 1.1286

In [5]:
from scripts.utils import print_run_status

selected_model_key = training_model_name("efficientnet", TRAINING_SAMPLING_MODE)
print_run_status(selected_model_key)

Model: efficientnet_undersampled
run_001: status=completed, epoch=10/10, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet_undersampled\runs\run_001


[{'run_id': 'run_001',
  'status': 'completed',
  'last_completed_epoch': 10,
  'target_epochs': 10,
  'checkpoint': 'yes',
  'best_model': 'yes',
  'output_dir': 'D:\\M L\\Diabetic_Retinopathy_Detection_mini_project\\models\\efficientnet_undersampled\\runs\\run_001'}]


## EfficientNetB0 — Baseline

| Parameter | Current Setting |
|---|---|
| Model Architecture | EfficientNetB0 |
| Pretrained Weights | ImageNet |
| Input Image Size | 300 × 300 × 3 |
| Number of Classes | 5 |
| Backbone Trainable | False (Frozen) |
| Optimizer | Adam |
| Learning Rate | 0.001 (1e-3) |
| Loss Function | Sparse Categorical Cross-Entropy |
| Training Metric | Sparse Categorical Accuracy |
| Dropout Rate | 0.30 (30%) |
| Classification Head | Global Average Pooling → Dropout → Dense (5, Softmax) |
| Class Weighting | None (Unweighted Baseline) |
| Batch Size | 4 (default; configurable per run) |
| Target Epochs | 10 (default; configurable per run) |
| Random Seed | 42 |
| Shuffle Buffer | 2048 |
| Best Model Selection | Lowest Validation Loss |


**Training data strategy:** the standalone `prepare_undersampled_split.ipynb` has already fixed the training subset at a maximum of 2,500 examples per class. Training scripts do not choose samples. Class weights (when enabled) are calculated from the manifest class counts. Validation and test are also fixed sampled subsets from the same 70:10:20 manifest.


## 6. Class-weighted EfficientNet — manually controlled training

**Run only after the unweighted EfficientNet baseline experiment you intend to compare against.** This is a separate experiment with its own run folders and checkpoints; it does not initialize from the trained unweighted model. It uses the same EfficientNetB0/ImageNet architecture and the same fixed `btgraham-300` split manifest.

Balanced class weights are calculated from the selected training subset only using `w_c = N / (K × n_c)`. They are applied to training examples only. Validation loss remains unweighted for model selection, and the test split is not evaluated here. Choose `scratch` for an independent new run or `resume` to continue the latest weighted-run checkpoint. The Epoch value is the **total target epoch**, not the number of additional epochs.


In [15]:
from scripts.train_efficientnet_weighted import train_efficientnet_weighted

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False

if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    print("1. Stop without training")
    print("2. Allow potentially slow CPU training")
    cpu_choice = int(input("Select CPU action [1-2]: ").strip())
    if cpu_choice == 2:
        allow_cpu = True
    elif cpu_choice != 1:
        raise ValueError("Choose 1 to stop or 2 to allow CPU training.")

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, then rerun this cell.")
else:
    print("1. Start a new scratch run")
    print("2. Resume the latest checkpointed run")
    run_choice = int(input("Select run mode [1-2]: ").strip())
    if run_choice not in (1, 2):
        raise ValueError("Choose 1 for scratch or 2 for resume.")
    run_mode = ("scratch", "resume")[run_choice - 1]

    batch_choices = (4, 8, 16, 32)
    for index, value in enumerate(batch_choices, start=1):
        print(f"{index}. Batch size {value}")
    batch_choice = int(input("Select batch size [1-4]: ").strip())
    if not 1 <= batch_choice <= len(batch_choices):
        raise ValueError("Choose one of the displayed batch-size options.")
    batch_size = batch_choices[batch_choice - 1]

    epochs_text = input("Enter total target epochs as a whole number [5]: ").strip()
    target_epochs = int(epochs_text or 5)
    if target_epochs < 1:
        raise ValueError("Target epochs must be at least 1.")

    learning_rate_text = input("Enter learning rate [0.001]: ").strip()
    learning_rate = float(learning_rate_text or 0.001)
    if learning_rate <= 0:
        raise ValueError("Learning rate must be greater than 0.")

    weighted_efficientnet_result = train_efficientnet_weighted(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate,
        training_sampling_mode=TRAINING_SAMPLING_MODE
    )


Training device(s): /physical_device:GPU:0
Starting a new, separate weighted run: efficientnet_weighted_undersampled/run_001
Using fixed 70:10:20 subset manifest: train=9,024, validation=1,289, test=2,578; SHA-256=d8a125deedf7…
Train     : 9,024 selected / 35,126 official; class counts=[2500, 2443, 2500, 873, 708]
Validation: 1,289 selected / 10,906 official; class counts=[961, 85, 187, 28, 28]
Test      : 2,578 selected / 42,670 official; class counts=[1897, 184, 380, 59, 58]
Training sampling mode: undersampled; cap: 2500
Input size: 300x300; classes: 5
Training-only class counts: {0: 2500, 1: 2443, 2: 2500, 3: 873, 4: 708}
Training-only class weights: {0: 0.72192, 1: 0.7387638149815801, 2: 0.72192, 3: 2.0673539518900346, 4: 2.549152542372881}
Validation weighting: none; best model selected by lowest validation loss
Starting at epoch 0; target total epochs: 5
Epoch 1/5
282/282 [==============================] - 56s 181ms/step - loss: 1.3900 - accuracy: 0.3454 - val_loss: 1.2917 - val

In [8]:
from scripts.utils import print_run_status

selected_model_key = training_model_name("efficientnet_weighted", TRAINING_SAMPLING_MODE)
print_run_status(selected_model_key)

Model: efficientnet_weighted_undersampled
run_001: status=interrupted, epoch=0/10, checkpoint=no, best_model=no
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet_weighted_undersampled\runs\run_001
run_002: status=completed, epoch=5/5, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\efficientnet_weighted_undersampled\runs\run_002


[{'run_id': 'run_001',
  'status': 'interrupted',
  'last_completed_epoch': 0,
  'target_epochs': 10,
  'checkpoint': 'no',
  'best_model': 'no',
  'output_dir': 'D:\\M L\\Diabetic_Retinopathy_Detection_mini_project\\models\\efficientnet_weighted_undersampled\\runs\\run_001'},
 {'run_id': 'run_002',
  'status': 'completed',
  'last_completed_epoch': 5,
  'target_epochs': 5,
  'checkpoint': 'yes',
  'best_model': 'yes',
  'output_dir': 'D:\\M L\\Diabetic_Retinopathy_Detection_mini_project\\models\\efficientnet_weighted_undersampled\\runs\\run_002'}]

### Class-weighted EfficientNet — current hyperparameters

| Parameter | Current Setting |
|---|---|
| Model Architecture | EfficientNetB0 |
| Pretrained Weights | ImageNet |
| Input Image Size | 300 × 300 × 3 |
| Number of Classes | 5 |
| Backbone Trainable | False (Frozen) |
| Optimizer | Adam |
| Learning Rate | 0.001 (1e-3) |
| Loss Function | Sparse Categorical Cross-Entropy with per-example sample weights |
| Training Metric | Sparse Categorical Accuracy (unweighted reporting metric) |
| Dropout Rate | 0.30 (30%) |
| Classification Head | Global Average Pooling → Dropout → Dense (5, Softmax) |
| Class Weighting | Balanced inverse-frequency: \(w_c = N/(K n_c)\), calculated from the selected training subset only |
| Batch Size | 4 (default; configurable per run) |
| Epoch | User-defined total target (prompt default: 10) |
| Random Seed | 42 |
| Shuffle Buffer | 2048 |
| Best Model Selection | Lowest unweighted validation loss |


**Training data strategy:** the standalone `prepare_undersampled_split.ipynb` has already fixed the training subset at a maximum of 2,500 examples per class. Training scripts do not choose samples. Class weights (when enabled) are calculated from the manifest class counts. Validation and test are also fixed sampled subsets from the same 70:10:20 manifest.


## 7. Weighted DenseNet121 — preview and training

This experiment uses a frozen ImageNet-pretrained DenseNet121 backbone, global average pooling, dropout, and a five-class softmax head. Balanced inverse-frequency class weights are computed from the selected **training subset only** and applied to training loss only. Validation remains unweighted and uses the saved 70:10:20 subset. It has independent checkpoints and run folders under the mode-specific model root (for example, `models/densenet_undersampled/`); it does not load EfficientNet weights. The total target Epoch defaults to 10 and batch size defaults to 4.

In [9]:
from scripts.train_densenet import train_densenet

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False

if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    print("1. Stop without training")
    print("2. Allow potentially slow CPU training")
    cpu_choice = int(input("Select CPU action [1-2]: ").strip())
    if cpu_choice == 2:
        allow_cpu = True
    elif cpu_choice != 1:
        raise ValueError("Choose 1 to stop or 2 to allow CPU training.")

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, then rerun this cell.")
else:
    print("1. Start a new scratch run")
    print("2. Resume the latest checkpointed run")
    run_choice = int(input("Select run mode [1-2]: ").strip())
    if run_choice not in (1, 2):
        raise ValueError("Choose 1 for scratch or 2 for resume.")
    run_mode = ("scratch", "resume")[run_choice - 1]

    batch_choices = (4, 8, 16, 32)
    for index, value in enumerate(batch_choices, start=1):
        print(f"{index}. Batch size {value}")
    batch_choice = int(input("Select batch size [1-4]: ").strip())
    if not 1 <= batch_choice <= len(batch_choices):
        raise ValueError("Choose one of the displayed batch-size options.")
    batch_size = batch_choices[batch_choice - 1]

    epochs_text = input("Enter total target epochs as a whole number [5]: ").strip()
    target_epochs = int(epochs_text or 5)
    if target_epochs < 1:
        raise ValueError("Target epochs must be at least 1.")

    learning_rate_text = input("Enter learning rate [0.001]: ").strip()
    learning_rate = float(learning_rate_text or 0.001)
    if learning_rate <= 0:
        raise ValueError("Learning rate must be greater than 0.")

    densenet_result = train_densenet(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate,
        training_sampling_mode=TRAINING_SAMPLING_MODE
    )


Model: "densenet121_dr_weighted"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 retinal_image (InputLayer)  [(None, 300, 300, 3)]     0         
                                                                 
 densenet_preprocessing (Lam  (None, 300, 300, 3)      0         
 bda)                                                            
                                                                 
 densenet121 (Functional)    (None, 9, 9, 1024)        7037504   
                                                                 
 global_average_pooling (Glo  (None, 1024)             0         
 balAveragePooling2D)                                            
                                                                 
 head_dropout (Dropout)      (None, 1024)              0         
                                                                 
 dr_severity (Dense)         (None, 5)     

### Weighted DenseNet — manually controlled training

Only run this cell when ready to train. If the model runs out of GPU memory, reduce batch size before changing the image size.

In [10]:
from scripts.train_densenet import train_densenet

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False

if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    print("1. Stop without training")
    print("2. Allow potentially slow CPU training")
    cpu_choice = int(input("Select CPU action [1-2]: ").strip())
    if cpu_choice == 2:
        allow_cpu = True
    elif cpu_choice != 1:
        raise ValueError("Choose 1 to stop or 2 to allow CPU training.")

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, then rerun this cell.")
else:
    print("1. Start a new scratch run")
    print("2. Resume the latest checkpointed run")
    run_choice = int(input("Select run mode [1-2]: ").strip())
    if run_choice not in (1, 2):
        raise ValueError("Choose 1 for scratch or 2 for resume.")
    run_mode = ("scratch", "resume")[run_choice - 1]

    batch_choices = (4, 8, 16, 32)
    for index, value in enumerate(batch_choices, start=1):
        print(f"{index}. Batch size {value}")
    batch_choice = int(input("Select batch size [1-4]: ").strip())
    if not 1 <= batch_choice <= len(batch_choices):
        raise ValueError("Choose one of the displayed batch-size options.")
    batch_size = batch_choices[batch_choice - 1]

    epochs_text = input("Enter total target epochs as a whole number [5]: ").strip()
    target_epochs = int(epochs_text or 5)
    if target_epochs < 1:
        raise ValueError("Target epochs must be at least 1.")

    learning_rate_text = input("Enter learning rate [0.001]: ").strip()
    learning_rate = float(learning_rate_text or 0.001)
    if learning_rate <= 0:
        raise ValueError("Learning rate must be greater than 0.")

    densenet_result = train_densenet(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate,
        training_sampling_mode=TRAINING_SAMPLING_MODE
    )


Training device(s): /physical_device:GPU:0
Starting a new, separate run: densenet_undersampled/run_001
Using fixed 70:10:20 subset manifest: train=9,024, validation=1,289, test=2,578; SHA-256=d8a125deedf7…
Train     : 9,024 selected / 35,126 official; class counts=[2500, 2443, 2500, 873, 708]
Validation: 1,289 selected / 10,906 official; class counts=[961, 85, 187, 28, 28]
Test      : 2,578 selected / 42,670 official; class counts=[1897, 184, 380, 59, 58]
Training sampling mode: undersampled; cap: 2500
Input size: 300x300; classes: 5
Training-only class counts: {0: 2500, 1: 2443, 2: 2500, 3: 873, 4: 708}
Training-only class weights: {0: 0.72192, 1: 0.7387638149815801, 2: 0.72192, 3: 2.0673539518900346, 4: 2.549152542372881}
Validation remains unweighted; best model selected by lowest validation loss.
Starting at epoch 0; total target epochs: 5
Epoch 1/5
282/282 [==============================] - 97s 299ms/step - loss: 1.6071 - accuracy: 0.2732 - val_loss: 1.3045 - val_accuracy: 0.5159


### Weighted DenseNet — inspect run status

Lists only this model's runs; it does not load a model or evaluate it.

In [11]:
from scripts.utils import print_run_status

selected_model_key = training_model_name("densenet", TRAINING_SAMPLING_MODE)
print_run_status(selected_model_key)

Model: densenet_undersampled
run_001: status=completed, epoch=5/5, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\densenet_undersampled\runs\run_001


[{'run_id': 'run_001',
  'status': 'completed',
  'last_completed_epoch': 5,
  'target_epochs': 5,
  'checkpoint': 'yes',
  'best_model': 'yes',
  'output_dir': 'D:\\M L\\Diabetic_Retinopathy_Detection_mini_project\\models\\densenet_undersampled\\runs\\run_001'}]

### Weighted DenseNet121 — hyperparameters

| Parameter | Current Setting |
|---|---|
| Model Architecture | DenseNet121 |
| Pretrained Weights | ImageNet |
| Input Image Size | 300 × 300 × 3 |
| Number of Classes | 5 |
| Backbone Trainable | False (Frozen) |
| Optimizer | Adam |
| Learning Rate | 0.001 (1e-3) |
| Loss Function | Sparse Categorical Cross-Entropy with training sample weights |
| Training Metric | Sparse Categorical Accuracy (unweighted reporting metric) |
| Dropout Rate | 0.30 (30%) |
| Classification Head | DenseNet preprocessing → Global Average Pooling → Dropout → Dense (5, Softmax) |
| Class Weighting | Balanced inverse-frequency: `N / (K × n_c)`, selected training subset only |
| Batch Size | 4 (default; configurable per run) |
| Epoch | User-defined total target (prompt default: 10) |
| Random Seed | 42 |
| Shuffle Buffer | 2048 |
| Best Model Selection | Lowest unweighted validation loss |

## 8. EfficientNetB0 + DenseNet121 feature-fusion model

The starter fusion experiment concatenates the two branches' global-average-pooled features, followed by dropout and a five-class softmax head. Both ImageNet feature extractors are frozen for this first resource-conscious version. This is **feature concatenation**, not attention fusion and not averaging final predictions. This initial fusion run is unweighted. Because it contains two backbones, batch size 4 may exceed 4 GB VRAM; reduce the batch size first if needed.

In [12]:
from scripts.train_fusion import train_fusion

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False

if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    print("1. Stop without training")
    print("2. Allow potentially slow CPU training")
    cpu_choice = int(input("Select CPU action [1-2]: ").strip())
    if cpu_choice == 2:
        allow_cpu = True
    elif cpu_choice != 1:
        raise ValueError("Choose 1 to stop or 2 to allow CPU training.")

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, then rerun this cell.")
else:
    print("1. Start a new scratch run")
    print("2. Resume the latest checkpointed run")
    run_choice = int(input("Select run mode [1-2]: ").strip())
    if run_choice not in (1, 2):
        raise ValueError("Choose 1 for scratch or 2 for resume.")
    run_mode = ("scratch", "resume")[run_choice - 1]

    batch_choices = (4, 8, 16, 32)
    for index, value in enumerate(batch_choices, start=1):
        print(f"{index}. Batch size {value}")
    batch_choice = int(input("Select batch size [1-4]: ").strip())
    if not 1 <= batch_choice <= len(batch_choices):
        raise ValueError("Choose one of the displayed batch-size options.")
    batch_size = batch_choices[batch_choice - 1]

    epochs_text = input("Enter total target epochs as a whole number [5]: ").strip()
    target_epochs = int(epochs_text or 5)
    if target_epochs < 1:
        raise ValueError("Target epochs must be at least 1.")

    learning_rate_text = input("Enter learning rate [0.001]: ").strip()
    learning_rate = float(learning_rate_text or 0.001)
    if learning_rate <= 0:
        raise ValueError("Learning rate must be greater than 0.")

    fusion_result = train_fusion(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate,
        training_sampling_mode=TRAINING_SAMPLING_MODE
    )


Model: "efficientnet_densenet_fusion_dr"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 retinal_image (InputLayer)     [(None, 300, 300, 3  0           []                               
                                )]                                                                
                                                                                                  
 densenet_preprocessing (Lambda  (None, 300, 300, 3)  0          ['retinal_image[0][0]']          
 )                                                                                                
                                                                                                  
 efficientnetb0 (Functional)    (None, 10, 10, 1280  4049571     ['retinal_image[0][0]']          
                                )                                   

### Fusion — manually controlled training

Training is separate from the EfficientNet and DenseNet run roots. Use `scratch` for a new independent fusion run or `resume` to restore its own latest valid checkpoint.

In [13]:
from scripts.train_fusion import train_fusion

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False

if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    print("1. Stop without training")
    print("2. Allow potentially slow CPU training")
    cpu_choice = int(input("Select CPU action [1-2]: ").strip())
    if cpu_choice == 2:
        allow_cpu = True
    elif cpu_choice != 1:
        raise ValueError("Choose 1 to stop or 2 to allow CPU training.")

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, then rerun this cell.")
else:
    print("1. Start a new scratch run")
    print("2. Resume the latest checkpointed run")
    run_choice = int(input("Select run mode [1-2]: ").strip())
    if run_choice not in (1, 2):
        raise ValueError("Choose 1 for scratch or 2 for resume.")
    run_mode = ("scratch", "resume")[run_choice - 1]

    batch_choices = (4, 8, 16, 32)
    for index, value in enumerate(batch_choices, start=1):
        print(f"{index}. Batch size {value}")
    batch_choice = int(input("Select batch size [1-4]: ").strip())
    if not 1 <= batch_choice <= len(batch_choices):
        raise ValueError("Choose one of the displayed batch-size options.")
    batch_size = batch_choices[batch_choice - 1]

    epochs_text = input("Enter total target epochs as a whole number [5]: ").strip()
    target_epochs = int(epochs_text or 5)
    if target_epochs < 1:
        raise ValueError("Target epochs must be at least 1.")

    learning_rate_text = input("Enter learning rate [0.001]: ").strip()
    learning_rate = float(learning_rate_text or 0.001)
    if learning_rate <= 0:
        raise ValueError("Learning rate must be greater than 0.")

    fusion_result = train_fusion(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate,
        training_sampling_mode=TRAINING_SAMPLING_MODE
    )


Training device(s): /physical_device:GPU:0
Starting a new, separate run: fusion_undersampled/run_001
Using fixed 70:10:20 subset manifest: train=9,024, validation=1,289, test=2,578; SHA-256=d8a125deedf7…
Train     : 9,024 selected / 35,126 official; class counts=[2500, 2443, 2500, 873, 708]
Validation: 1,289 selected / 10,906 official; class counts=[961, 85, 187, 28, 28]
Test      : 2,578 selected / 42,670 official; class counts=[1897, 184, 380, 59, 58]
Class weighting: none; validation remains unweighted.
Training sampling mode: undersampled; cap: 2500
Input size: 300x300; classes: 5
Best weights are selected using unweighted validation loss.
Starting at epoch 0; total target epochs: 5
Epoch 1/5
282/282 [==============================] - 128s 419ms/step - loss: 1.4093 - accuracy: 0.3590 - val_loss: 1.0596 - val_accuracy: 0.5857
Epoch 2/5
282/282 [==============================] - 115s 404ms/step - loss: 1.2575 - accuracy: 0.4229 - val_loss: 1.2380 - val_accuracy: 0.3840
Epoch 3/5
282/

### Fusion — inspect run status

In [14]:
from scripts.utils import print_run_status

selected_model_key = training_model_name("fusion", TRAINING_SAMPLING_MODE)
print_run_status(selected_model_key)

Model: fusion_undersampled
run_001: status=completed, epoch=5/5, checkpoint=yes, best_model=yes
  Output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\models\fusion_undersampled\runs\run_001


[{'run_id': 'run_001',
  'status': 'completed',
  'last_completed_epoch': 5,
  'target_epochs': 5,
  'checkpoint': 'yes',
  'best_model': 'yes',
  'output_dir': 'D:\\M L\\Diabetic_Retinopathy_Detection_mini_project\\models\\fusion_undersampled\\runs\\run_001'}]

### Fusion — current hyperparameters

| Parameter | Current Setting |
|---|---|
| Model Architecture | EfficientNetB0 + DenseNet121 feature concatenation |
| Pretrained Weights | ImageNet for both backbones |
| Input Image Size | 300 × 300 × 3 |
| Number of Classes | 5 |
| Backbones Trainable | False (both frozen) |
| Optimizer | Adam |
| Learning Rate | 0.001 (1e-3) |
| Loss Function | Sparse Categorical Cross-Entropy |
| Training Metric | Sparse Categorical Accuracy |
| Dropout Rate | 0.30 (30%) |
| Classification Head | Concatenate pooled features → Dropout → Dense (5, Softmax) |
| Class Weighting | None (initial fusion experiment) |
| Batch Size | 4 (default; configurable per run) |
| Epoch | User-defined total target (prompt default: 10) |
| Random Seed | 42 |
| Shuffle Buffer | 2048 |
| Best Model Selection | Lowest unweighted validation loss |
